In [ ]:
from pyspark.sql import SparkSession
from pyspark.mllib.fpm import FPGrowth

# Load Data from item_sets.txt
print("Loading data from item_sets.txt...")
item_sets_data = spark.sparkContext.textFile("item_sets.txt")
item_sets_transactions = item_sets_data.map(lambda line: line.strip().split('\t'))

# Load Data from graph.net
print("Loading data from graph.net...")
graph_data = spark.sparkContext.textFile("graph.net")
graph_transactions = graph_data.map(lambda line: line.strip().split('\t'))

# Combine Data (if necessary)
# We can join item_sets_transactions with graph_transactions based on common transaction IDs.

# Task Decomposition: Partition the dataset
print("Partitioning the dataset...")
partitions = item_sets_transactions.randomSplit([0.25, 0.25, 0.25, 0.25])  # Divide into four partitions

# Parallelization: Apply FP-Growth algorithm on each partition
print("Applying FP-Growth algorithm...")
frequent_itemsets = []
for partition in partitions:
    model = FPGrowth.train(partition, minSupport=0.1)
    frequent_itemsets.append(model.freqItemsets())

# Combine Results
print("Combining results...")
all_frequent_itemsets = spark.sparkContext.union(frequent_itemsets)

# Output Results
print("Outputting frequent itemsets...")
for itemset in all_frequent_itemsets.collect():
    print(itemset)

# Stop Spark Session
spark.stop()



Loading data from item_sets.txt...


AttributeError: 'NoneType' object has no attribute 'sc'

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import collect_list

# Step 1: Initialize Spark Session
spark = SparkSession.builder \
    .appName("Association Rule Mining") \
    .getOrCreate()

# Step 2: Load Data
item_sets_data = spark.read.csv("item_sets.txt", sep="\t", header=False)
graph_data = spark.read.csv("graph.net", sep="\t", header=False)

# Step 3: Preprocess Data
# Assuming item_sets_data is already in the correct format

# Convert graph_data to the same format as item_sets_data
graph_data = graph_data.groupBy("_c0").agg(collect_list("_c1").alias("items"))

# Step 4: Train FP-Growth Model
# Combine item sets data and preprocessed graph data
transaction_data = item_sets_data.union(graph_data)

# Step 5: Train FP-Growth Model
# Train FP-Growth model
fp_growth = FPGrowth(itemsCol="items", minSupport=0.1, minConfidence=0.5)
model = fp_growth.fit(transaction_data)

# Step 6: Generate Association Rules
association_rules = model.associationRules

# Step 7: Evaluate and Interpret Results
print("Association Rules:")
association_rules.show()

# Stop Spark Session
spark.stop()


AnalysisException: [NUM_COLUMNS_MISMATCH] UNION can only be performed on inputs with the same number of columns, but the first input has 6 columns and the second input has 2 columns.;
'Union false, false
:- Relation [_c0#95,_c1#96,_c2#97,_c3#98,_c4#99,_c5#100] csv
+- Aggregate [_c0#124], [_c0#124, collect_list(_c1#125, 0, 0) AS items#131]
   +- Relation [_c0#124,_c1#125] csv


In [ ]:
# Print schema of item_sets_data DataFrame
print("Schema of item_sets_data:")
item_sets_data.printSchema()

# Print schema of graph_data DataFrame
print("Schema of graph_data:")
graph_data.printSchema()



Schema of item_sets_data:
root
 |-- _c0: string (nullable = true)
 |-- _c1: string (nullable = true)
 |-- _c2: string (nullable = true)
 |-- _c3: string (nullable = true)
 |-- _c4: string (nullable = true)
 |-- _c5: string (nullable = true)

Schema of graph_data:
root
 |-- _c0: string (nullable = true)
 |-- items: array (nullable = false)
 |    |-- element: string (containsNull = false)



In [ ]:
from pyspark.sql.functions import array

# Preprocess item_sets_data DataFrame to match the structure of graph_data
item_sets_data = item_sets_data.withColumn("items", array("_c1", "_c2", "_c3", "_c4", "_c5"))

# Select relevant columns from graph_data DataFrame
graph_data = graph_data.select("_c0", "items")


In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import lit

# Initialize Spark session
spark = SparkSession.builder \
    .appName("AssociationRuleMining") \
    .getOrCreate()

# Load item_sets.txt and graph.net datasets into Spark DataFrames
item_sets_data = spark.read.csv("item_sets.txt", sep="\t")
graph_data = spark.read.csv("graph.net", sep="\t")

# Preprocess item_sets_data to match the structure of graph_data
item_sets_data = item_sets_data.selectExpr("_c0 as _c0", "array(_c1, _c2, _c3, _c4, _c5) as items")

# Select relevant columns from graph_data
graph_data = graph_data.selectExpr("_c0 as _c0")

# Add a dummy column to graph_data to match the number of columns in item_sets_data
graph_data = graph_data.withColumn("_c1", lit(None))

# Union item_sets_data and graph_data
transaction_data = item_sets_data.union(graph_data)



In [ ]:
# Read the item_sets.txt file and extract the last value of each line
with open('item_sets.txt', 'r') as f:
    lines = f.readlines()

# Extract the last value of each line to create the target variable
target_values = [line.strip().split()[-1] for line in lines]

# Print the first few target values for verification
print("Target values:", target_values[:100])


Target values: ['376927', '489980', '489980', '268405', '410518', '412978', '260929', '328722', '491602', '209079', '283396', '245641', '419049', '417202', '360574', '368935', '455493', '430267', '430267', '481432', '366493', '100302', '369148', '461186', '375651', '118574', '383030', '383030', '262922', '447269', '376346', '219140', '100109', '428135', '100111', '288579', '492817', '515230', '439430', '139183', '277499', '215331', '157330', '519875', '515277', '429311', '454247', '482325', '385843', '233447', '369070', '378845', '408460', '292086', '102946', '490912', '528646', '528646', '528646', '516521', '420108', '287628', '420098', '362792', '303083', '525758', '353008', '441931', '365721', '314074', '374826', '133716', '114730', '250089', '272915', '394341', '483498', '280056', '58695', '58695', '328616', '363501', '143831', '448940', '448940', '311202', '158198', '171296', '519723', '519722', '519722', '543957', '542532', '273940', '438065', '311456', '539828', '158198', '54567

In [ ]:
# Read the graph.net file and extract relationships between IDs
relationships = {}
with open('graph.net', 'r') as f:
    lines = f.readlines()
    for line in lines:
        line = line.strip().split()
        if len(line) == 2:  # Ensure the line contains two IDs
            source_id, target_id = line
            if source_id not in relationships:
                relationships[source_id] = []
            if target_id not in relationships:
                relationships[target_id] = []
            relationships[source_id].append(target_id)
            relationships[target_id].append(source_id)  # Consider A -> B and B -> A

# Print the relationships for verification
print("Relationships:")
print(relationships)

# Read the item_sets.txt file and extract the last value from each line as the target value
targets = []
with open('item_sets.txt', 'r') as f:
    lines = f.readlines()
    for line in lines:
        line = line.strip().split()
        target = line[-1]  # Last value in the line
        targets.append(target)

# Print the first few target values for verification
print("Target values:")
print(targets[:5])

# Define a function to predict the target value based on relationships
def predict_target(person_id):
    if person_id in relationships:
        friends = relationships[person_id]
        # Implement your prediction logic here
        # For example, you can count the occurrences of each friend's ID as the target value
        # and return the most frequent one
        target_prediction = max(set(friends), key=friends.count)
        return target_prediction
    else:
        return None  # Return None if no relationships found for the person

# Predict target values for each person
predicted_targets = [predict_target(person_id) for person_id in targets]

# Print the first few predicted target values for verification
print("Predicted target values:")
print(predicted_targets[:5])


IOPub data rate exceeded.
The notebook server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--NotebookApp.iopub_data_rate_limit`.

Current values:
NotebookApp.iopub_data_rate_limit=1000000.0 (bytes/sec)
NotebookApp.rate_limit_window=3.0 (secs)



Predicted target values:
['202036', '479170', '479170', '118111', '202812']


In [ ]:
# Load item_sets.txt and graph.net
with open('item_sets.txt', 'r') as f:
    item_sets_data = [line.strip().split('\t') for line in f.readlines()]

with open('graph.net', 'r') as f:
    graph_data = [line.strip().split('\t') for line in f.readlines()]

# Extract target values from item_sets_data (last item in each sequence)
targets = [sequence[-1] for sequence in item_sets_data]

# Create bidirectional associations from graph_data
associations = {}
for line in graph_data:
    source = line[0]
    target = line[1]
    if source not in associations:
        associations[source] = set()
    associations[source].add(target)
    if target not in associations:
        associations[target] = set()
    associations[target].add(source)

# Now associations contains bidirectional associations between items


In [ ]:
from mlxtend.preprocessing import TransactionEncoder
from mlxtend.frequent_patterns import apriori
import pandas as pd

# Load the item sets data
print("Loading item sets data...")
item_sets_file = 'item_sets.txt'
with open(item_sets_file, 'r') as file:
    lines = file.readlines()

# Extract item sets from the file
item_sets = [line.strip().split('\t') for line in lines]

# Create a TransactionEncoder and transform the data
print("Transforming item sets...")
te = TransactionEncoder()
te_ary = te.fit(item_sets).transform(item_sets)
df = pd.DataFrame(te_ary, columns=te.columns_)

# Perform association rule mining
print("Performing association rule mining...")
frequent_itemsets = apriori(df, min_support=0.001, use_colnames=True)

# Load the graph data
print("Loading graph data...")
graph_file = 'graph.net'
with open(graph_file, 'r') as file:
    graph_lines = file.readlines()

# Extract edges from the graph file
edges = [line.strip().split('\t') for line in graph_lines]

# Create a set of frequent items
frequent_items = set(df.columns)

# Create a dictionary to store frequent item pairs and their frequencies
item_pairs = {}
for edge in edges:
    item1, item2 = edge
    if item1 in frequent_items and item2 in frequent_items:
        pair = tuple(sorted([item1, item2]))
        item_pairs[pair] = item_pairs.get(pair, 0) + 1

# Filter frequent item pairs based on support threshold
min_support_count = int(len(item_sets) * 0.001)
frequent_pairs = {pair for pair, count in item_pairs.items() if count >= min_support_count}

# Predict the most appropriate item for each item set
print("Predicting the most appropriate item for each item set...")
predictions = []
for i, item_set in enumerate(item_sets, start=1):
    max_confidence = 0
    best_item = None
    for item in item_set:
        for pair in frequent_pairs:
            if item in pair:
                other_item = pair[0] if item == pair[1] else pair[1]
                confidence = item_pairs.get(pair, 0) / item_sets.count(item_set)
                if confidence > max_confidence:
                    max_confidence = confidence
                    best_item = other_item
    predictions.append(best_item)

# Create a DataFrame for the submission file
print("Creating submission file...")
submission = pd.DataFrame({'id': range(1, len(predictions) + 1), 'target': predictions})

# Write the submission file to CSV
submission.to_csv('submission.csv', index=False)

print("Submission file 'submission.csv' generated successfully.")


/usr/local/lib/python3.10/dist-packages/ipykernel/ipkernel.py:283: DeprecationWarning: `should_run_async` will not call `transform_cell` automatically in the future. Please pass the result to `transformed_cell` argument and any exception that happen during thetransform in `preprocessing_exc_tuple` in IPython 7.17 and above.
  and should_run_async(code)


Loading item sets data...
Transforming item sets...
Performing association rule mining...
Loading graph data...
Predicting the most appropriate item for each item set...
Creating submission file...
Submission file 'submission.csv' generated successfully.


In [ ]:
from google.colab import files

# Assuming the CSV file is saved as 'submission.csv' in the current working directory
files.download('submission.csv')


/usr/local/lib/python3.10/dist-packages/ipykernel/ipkernel.py:283: DeprecationWarning: `should_run_async` will not call `transform_cell` automatically in the future. Please pass the result to `transformed_cell` argument and any exception that happen during thetransform in `preprocessing_exc_tuple` in IPython 7.17 and above.
  and should_run_async(code)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
# Predict the most appropriate item for each item set
print("Predicting the most appropriate item for each item set...")
predictions = []
for i, item_set in enumerate(item_sets, start=1):
    max_confidence = 0
    best_item = None
    for item in item_set:
        for pair in frequent_pairs:
            if item in pair:
                other_item = pair[0] if item == pair[1] else pair[1]
                confidence = item_pairs.get(pair, 0) / item_sets.count(item_set)
                if confidence > max_confidence:
                    max_confidence = confidence
                    best_item = other_item
    predictions.append(best_item)
    print(f"Processed item set {i}/{len(item_sets)}")

print("Predictions:", predictions)


/usr/local/lib/python3.10/dist-packages/ipykernel/ipkernel.py:283: DeprecationWarning: `should_run_async` will not call `transform_cell` automatically in the future. Please pass the result to `transformed_cell` argument and any exception that happen during thetransform in `preprocessing_exc_tuple` in IPython 7.17 and above.
  and should_run_async(code)


Streaming output truncated to the last 5000 lines.
Processed item set 70151/75149
Processed item set 70152/75149
Processed item set 70153/75149
Processed item set 70154/75149
Processed item set 70155/75149
Processed item set 70156/75149
Processed item set 70157/75149
Processed item set 70158/75149
Processed item set 70159/75149
Processed item set 70160/75149
Processed item set 70161/75149
Processed item set 70162/75149
Processed item set 70163/75149
Processed item set 70164/75149
Processed item set 70165/75149
Processed item set 70166/75149
Processed item set 70167/75149
Processed item set 70168/75149
Processed item set 70169/75149
Processed item set 70170/75149
Processed item set 70171/75149
Processed item set 70172/75149
Processed item set 70173/75149
Processed item set 70174/75149
Processed item set 70175/75149
Processed item set 70176/75149
Processed item set 70177/75149
Processed item set 70178/75149
Processed item set 70179/75149
Processed item set 70180/75149
Processed item set 

In [ ]:
import pandas as pd

# Load the item sets data
print("Loading item sets data...")
item_sets_file = 'item_sets.txt'
with open(item_sets_file, 'r') as file:
    item_sets = [line.strip().split('\t') for line in file]

# Preprocess the graph data to create a dictionary of neighbors for each item
print("Preprocessing graph data...")
neighbors = {}
graph_file = 'graph.net'
with open(graph_file, 'r') as file:
    for line in file:
        node1, node2 = line.strip().split('\t')
        neighbors.setdefault(node1, set()).add(node2)
        neighbors.setdefault(node2, set()).add(node1)

# Predict the most appropriate item for each item set
print("Predicting the most appropriate item for each item set...")
predictions = []
for i, item_set in enumerate(item_sets):
    candidate_neighbors = {}
    for item in item_set:
        if item in neighbors:
            for neighbor in neighbors[item]:
                if neighbor not in item_set:
                    candidate_neighbors[neighbor] = candidate_neighbors.get(neighbor, 0) + 1
    if candidate_neighbors:
        best_neighbor = max(candidate_neighbors, key=candidate_neighbors.get)
        predictions.append(best_neighbor)
    else:
        predictions.append(None)
    print(f"Processed item set {i + 1}/{len(item_sets)}")

# Create a DataFrame for the submission file
print("Creating submission file...")
submission = pd.DataFrame({'id': range(1, len(predictions) + 1), 'target': predictions})

# Write the submission file to CSV
submission.to_csv('submission.csv', index=False)

print("Submission file 'submission.csv' generated successfully.")


/usr/local/lib/python3.10/dist-packages/ipykernel/ipkernel.py:283: DeprecationWarning: `should_run_async` will not call `transform_cell` automatically in the future. Please pass the result to `transformed_cell` argument and any exception that happen during thetransform in `preprocessing_exc_tuple` in IPython 7.17 and above.
  and should_run_async(code)


Streaming output truncated to the last 5000 lines.
Processed item set 70152/75149
Processed item set 70153/75149
Processed item set 70154/75149
Processed item set 70155/75149
Processed item set 70156/75149
Processed item set 70157/75149
Processed item set 70158/75149
Processed item set 70159/75149
Processed item set 70160/75149
Processed item set 70161/75149
Processed item set 70162/75149
Processed item set 70163/75149
Processed item set 70164/75149
Processed item set 70165/75149
Processed item set 70166/75149
Processed item set 70167/75149
Processed item set 70168/75149
Processed item set 70169/75149
Processed item set 70170/75149
Processed item set 70171/75149
Processed item set 70172/75149
Processed item set 70173/75149
Processed item set 70174/75149
Processed item set 70175/75149
Processed item set 70176/75149
Processed item set 70177/75149
Processed item set 70178/75149
Processed item set 70179/75149
Processed item set 70180/75149
Processed item set 70181/75149
Processed item set 

In [ ]:
import pandas as pd

# Load the item sets data
print("Loading item sets data...")
item_sets_file = 'item_sets.txt'
with open(item_sets_file, 'r') as file:
    item_sets = [line.strip().split('\t') for line in file]

# Preprocess the graph data to create a dictionary of neighbors for each item
print("Preprocessing graph data...")
neighbors = {}
graph_file = 'graph.net'
with open(graph_file, 'r') as file:
    for line in file:
        node1, node2 = line.strip().split('\t')
        neighbors.setdefault(node1, set()).add(node2)
        neighbors.setdefault(node2, set()).add(node1)

# Predict the most appropriate item for each item set
print("Predicting the most appropriate item for each item set...")
predictions = []
num_processed = 0
for i, item_set in enumerate(item_sets):
    candidate_neighbors = {}
    for item in item_set:
        if item in neighbors:
            for neighbor in neighbors[item]:
                if neighbor not in item_set:
                    candidate_neighbors[neighbor] = candidate_neighbors.get(neighbor, 0) + 1
    if candidate_neighbors:
        best_neighbor = max(candidate_neighbors, key=candidate_neighbors.get)
        predictions.append(best_neighbor)
    else:
        predictions.append(None)
    num_processed += 1
    print(f"Processed item set {num_processed}/{len(item_sets)}")

# Create a DataFrame for the submission file
print("Creating submission file...")
submission = pd.DataFrame({'id': range(1, len(predictions) + 1), 'target': predictions})

# Write the submission file to CSV
submission.to_csv('submission.csv', index=False)

print("Submission file 'submission.csv' generated successfully.")


/usr/local/lib/python3.10/dist-packages/ipykernel/ipkernel.py:283: DeprecationWarning: `should_run_async` will not call `transform_cell` automatically in the future. Please pass the result to `transformed_cell` argument and any exception that happen during thetransform in `preprocessing_exc_tuple` in IPython 7.17 and above.
  and should_run_async(code)


Streaming output truncated to the last 5000 lines.
Processed item set 70152/75149
Processed item set 70153/75149
Processed item set 70154/75149
Processed item set 70155/75149
Processed item set 70156/75149
Processed item set 70157/75149
Processed item set 70158/75149
Processed item set 70159/75149
Processed item set 70160/75149
Processed item set 70161/75149
Processed item set 70162/75149
Processed item set 70163/75149
Processed item set 70164/75149
Processed item set 70165/75149
Processed item set 70166/75149
Processed item set 70167/75149
Processed item set 70168/75149
Processed item set 70169/75149
Processed item set 70170/75149
Processed item set 70171/75149
Processed item set 70172/75149
Processed item set 70173/75149
Processed item set 70174/75149
Processed item set 70175/75149
Processed item set 70176/75149
Processed item set 70177/75149
Processed item set 70178/75149
Processed item set 70179/75149
Processed item set 70180/75149
Processed item set 70181/75149
Processed item set 

In [ ]:
import pandas as pd

# Load the item sets data
print("Loading item sets data...")
with open('item_sets.txt', 'r') as file:
    item_sets = [line.strip().split('\t') for line in file]

print("Item sets data loaded successfully.")

# Preprocess the graph data to create a dictionary of neighbors for each item
print("Preprocessing graph data...")
neighbors = {}
with open('graph.net', 'r') as file:
    for line in file:
        node1, node2 = line.strip().split('\t')
        neighbors.setdefault(node1, set()).add(node2)
        neighbors.setdefault(node2, set()).add(node1)

print("Graph data preprocessed successfully.")

# Predict the most appropriate item for each item set
print("Predicting the most appropriate item for each item set...")
predictions = []
for i, item_set in enumerate(item_sets):
    candidate_neighbors = {}
    for item in item_set:
        candidate_neighbors.update({neighbor: candidate_neighbors.get(neighbor, 0) + 1 for neighbor in neighbors.get(item, []) if neighbor not in item_set})
    best_neighbor = max(candidate_neighbors, key=candidate_neighbors.get, default=None)
    predictions.append(best_neighbor)
    print(f"Processed item set {i + 1}/{len(item_sets)}")

print("Prediction completed.")

# Create a DataFrame for the submission file
print("Creating submission file...")
submission = pd.DataFrame({'id': range(1, len(predictions) + 1), 'target': predictions})

# Write the submission file to CSV
submission.to_csv('submission.csv', index=False)

print("Submission file 'submission.csv' generated successfully.")


/usr/local/lib/python3.10/dist-packages/ipykernel/ipkernel.py:283: DeprecationWarning: `should_run_async` will not call `transform_cell` automatically in the future. Please pass the result to `transformed_cell` argument and any exception that happen during thetransform in `preprocessing_exc_tuple` in IPython 7.17 and above.
  and should_run_async(code)


Streaming output truncated to the last 5000 lines.
Processed item set 70153/75149
Processed item set 70154/75149
Processed item set 70155/75149
Processed item set 70156/75149
Processed item set 70157/75149
Processed item set 70158/75149
Processed item set 70159/75149
Processed item set 70160/75149
Processed item set 70161/75149
Processed item set 70162/75149
Processed item set 70163/75149
Processed item set 70164/75149
Processed item set 70165/75149
Processed item set 70166/75149
Processed item set 70167/75149
Processed item set 70168/75149
Processed item set 70169/75149
Processed item set 70170/75149
Processed item set 70171/75149
Processed item set 70172/75149
Processed item set 70173/75149
Processed item set 70174/75149
Processed item set 70175/75149
Processed item set 70176/75149
Processed item set 70177/75149
Processed item set 70178/75149
Processed item set 70179/75149
Processed item set 70180/75149
Processed item set 70181/75149
Processed item set 70182/75149
Processed item set 

In [ ]:
import pandas as pd
import networkx as nx

# Load the item sets data
print("Loading item sets data...")
with open('item_sets.txt', 'r') as file:
    item_sets = [line.strip().split('\t') for line in file]

print("Item sets data loaded successfully.")

# Preprocess the graph data to create a networkx graph
print("Preprocessing graph data...")
G = nx.Graph()
with open('graph.net', 'r') as file:
    for line in file:
        node1, node2 = line.strip().split('\t')
        G.add_edge(node1, node2)

print("Graph data preprocessed successfully.")

# Calculate betweenness centrality for each node
print("Calculating betweenness centrality...")
betweenness = nx.betweenness_centrality(G)

print("Betweenness centrality calculated successfully.")

# Predict the most appropriate item for each item set
print("Predicting the most appropriate item for each item set...")
predictions = []
for i, item_set in enumerate(item_sets):
    max_betweenness = 0
    best_neighbor = None
    for item in item_set:
        neighbors = G.neighbors(item)
        for neighbor in neighbors:
            if neighbor not in item_set:
                if betweenness[neighbor] > max_betweenness:
                    max_betweenness = betweenness[neighbor]
                    best_neighbor = neighbor
    predictions.append(best_neighbor)
    print(f"Processed item set {i + 1}/{len(item_sets)}")

print("Prediction completed.")

# Create a DataFrame for the submission file
print("Creating submission file...")
submission = pd.DataFrame({'id': range(1, len(predictions) + 1), 'target': predictions})

# Write the submission file to CSV
submission.to_csv('submission.csv', index=False)

print("Submission file 'submission.csv' generated successfully.")


Loading item sets data...
Item sets data loaded successfully.
Preprocessing graph data...
Graph data preprocessed successfully.
Calculating betweenness centrality...


In [ ]:
import pandas as pd
import networkx as nx

# Load the item sets data
print("Loading item sets data...")
with open('item_sets.txt', 'r') as file:
    item_sets = [line.strip().split('\t') for line in file]

print("Item sets data loaded successfully.")

# Preprocess the graph data to create a networkx graph
print("Preprocessing graph data...")
G = nx.Graph()
with open('graph.net', 'r') as file:
    for line in file:
        node1, node2 = line.strip().split('\t')
        G.add_edge(node1, node2)

print("Graph data preprocessed successfully.")

# Detect communities using Label Propagation method
print("Detecting communities...")
communities = list(nx.algorithms.community.label_propagation.asyn_lpa_communities(G))

print("Communities detected successfully.")

# Create community dictionary
community_dict = {}
for i, community in enumerate(communities):
    for node in community:
        community_dict[node] = i

# Predict the most appropriate item for each item set
print("Predicting the most appropriate item for each item set...")
predictions = []
for i, item_set in enumerate(item_sets):
    community_items = set()
    for item in item_set:
        community_items.add(community_dict.get(item, -1))

    # Find the most common community in the item set
    predicted_community = max(community_items, key=community_items.count)

    # Find the most common item in the predicted community
    predicted_items = [node for node, community in community_dict.items() if community == predicted_community]
    prediction = max(predicted_items, key=G.degree)
    predictions.append(prediction)
    print(f"Processed item set {i + 1}/{len(item_sets)}")

print("Prediction completed.")

# Create a DataFrame for the submission file
print("Creating submission file...")
submission = pd.DataFrame({'id': range(1, len(predictions) + 1), 'target': predictions})

# Write the submission file to CSV
submission.to_csv('submission.csv', index=False)

print("Submission file 'submission.csv' generated successfully.")


Loading item sets data...
Item sets data loaded successfully.
Preprocessing graph data...
Graph data preprocessed successfully.
Detecting communities...
Communities detected successfully.
Predicting the most appropriate item for each item set...


AttributeError: 'set' object has no attribute 'count'

In [ ]:
# Predict the most appropriate item for each item set
print("Predicting the most appropriate item for each item set...")
predictions = []
for i, item_set in enumerate(item_sets):
    community_items = []
    for item in item_set:
        community_items.append(community_dict.get(item, -1))

    # Find the most common community in the item set
    predicted_community = max(community_items, key=community_items.count)

    # Find the most common item in the predicted community
    predicted_items = [node for node, community in community_dict.items() if community == predicted_community]
    prediction = max(predicted_items, key=G.degree)
    predictions.append(prediction)
    print(f"Processed item set {i + 1}/{len(item_sets)}")

print("Prediction completed.")

# Create a DataFrame for the submission file
print("Creating submission file...")
submission = pd.DataFrame({'id': range(1, len(predictions) + 1), 'target': predictions})

# Write the submission file to CSV
submission.to_csv('submission.csv', index=False)

print("Submission file 'submission.csv' generated successfully.")


Streaming output truncated to the last 5000 lines.
Processed item set 70153/75149
Processed item set 70154/75149
Processed item set 70155/75149
Processed item set 70156/75149
Processed item set 70157/75149
Processed item set 70158/75149
Processed item set 70159/75149
Processed item set 70160/75149
Processed item set 70161/75149
Processed item set 70162/75149
Processed item set 70163/75149
Processed item set 70164/75149
Processed item set 70165/75149
Processed item set 70166/75149
Processed item set 70167/75149
Processed item set 70168/75149
Processed item set 70169/75149
Processed item set 70170/75149
Processed item set 70171/75149
Processed item set 70172/75149
Processed item set 70173/75149
Processed item set 70174/75149
Processed item set 70175/75149
Processed item set 70176/75149
Processed item set 70177/75149
Processed item set 70178/75149
Processed item set 70179/75149
Processed item set 70180/75149
Processed item set 70181/75149
Processed item set 70182/75149
Processed item set 

In [ ]:
from google.colab import files

# Assuming the CSV file is saved as 'submission.csv' in the current working directory
files.download('submission.csv')


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
import pandas as pd

# Load item sets data
print("Loading item sets data...")
with open('item_sets.txt', 'r') as file:
    item_sets = [line.strip().split('\t') for line in file]

print("Item sets data loaded successfully.")

# Preprocess graph data to create a dictionary of neighbors for each item
print("Preprocessing graph data...")
neighbors = {}
with open('graph.net', 'r') as file:
    for line in file:
        node1, node2 = line.strip().split('\t')
        neighbors.setdefault(node1, set()).add(node2)
        neighbors.setdefault(node2, set()).add(node1)

print("Graph data preprocessed successfully.")

# Predict the most appropriate item for each item set
print("Predicting the most appropriate item for each item set...")
predictions = []
for i, item_set in enumerate(item_sets):
    candidate_neighbors = {}
    for item in item_set:
        for neighbor in neighbors.get(item, []):
            if neighbor not in item_set:
                candidate_neighbors[neighbor] = candidate_neighbors.get(neighbor, 0) + 1
    best_neighbor = max(candidate_neighbors, key=candidate_neighbors.get, default=None)
    predictions.append(best_neighbor)
    print(f"Processed item set {i + 1}/{len(item_sets)}")

print("Prediction completed.")

# Create DataFrame for the submission file
print("Creating submission file...")
submission = pd.DataFrame({'id': range(1, len(predictions) + 1), 'target': predictions})

# Write the submission file to CSV
submission.to_csv('submission.csv', index=False)

print("Submission file 'submission.csv' generated successfully.")


Streaming output truncated to the last 5000 lines.
Processed item set 70153/75149
Processed item set 70154/75149
Processed item set 70155/75149
Processed item set 70156/75149
Processed item set 70157/75149
Processed item set 70158/75149
Processed item set 70159/75149
Processed item set 70160/75149
Processed item set 70161/75149
Processed item set 70162/75149
Processed item set 70163/75149
Processed item set 70164/75149
Processed item set 70165/75149
Processed item set 70166/75149
Processed item set 70167/75149
Processed item set 70168/75149
Processed item set 70169/75149
Processed item set 70170/75149
Processed item set 70171/75149
Processed item set 70172/75149
Processed item set 70173/75149
Processed item set 70174/75149
Processed item set 70175/75149
Processed item set 70176/75149
Processed item set 70177/75149
Processed item set 70178/75149
Processed item set 70179/75149
Processed item set 70180/75149
Processed item set 70181/75149
Processed item set 70182/75149
Processed item set 

In [ ]:
import pandas as pd

# Load item sets data
print("Loading item sets data...")
with open('item_sets.txt', 'r') as file:
    item_sets = [line.strip().split('\t') for line in file]

print("Item sets data loaded successfully.")

# Preprocess graph data to create a dictionary of neighbors for each item
print("Preprocessing graph data...")
neighbors = {}
with open('graph.net', 'r') as file:
    for line in file:
        node1, node2 = line.strip().split('\t')
        neighbors.setdefault(node1, set()).add(node2)
        neighbors.setdefault(node2, set()).add(node1)

print("Graph data preprocessed successfully.")

# Predict the most appropriate item for each item set
print("Predicting the most appropriate item for each item set...")
predictions = []
for i, item_set in enumerate(item_sets):
    candidate_neighbors = {}
    for item in item_set:
        for neighbor in neighbors.get(item, []):
            if neighbor not in item_set:
                candidate_neighbors[neighbor] = candidate_neighbors.get(neighbor, 0) + 1
    sorted_candidates = sorted(candidate_neighbors.items(), key=lambda x: x[1], reverse=True)
    best_neighbor = sorted_candidates[0][0] if sorted_candidates else None
    predictions.append(best_neighbor)
    print(f"Processed item set {i + 1}/{len(item_sets)}")

print("Prediction completed.")

# Create DataFrame for the submission file
print("Creating submission file...")
submission = pd.DataFrame({'id': range(1, len(predictions) + 1), 'target': predictions})

# Write the submission file to CSV
submission.to_csv('submission.csv', index=False)

print("Submission file 'submission.csv' generated successfully.")


Streaming output truncated to the last 5000 lines.
Processed item set 70153/75149
Processed item set 70154/75149
Processed item set 70155/75149
Processed item set 70156/75149
Processed item set 70157/75149
Processed item set 70158/75149
Processed item set 70159/75149
Processed item set 70160/75149
Processed item set 70161/75149
Processed item set 70162/75149
Processed item set 70163/75149
Processed item set 70164/75149
Processed item set 70165/75149
Processed item set 70166/75149
Processed item set 70167/75149
Processed item set 70168/75149
Processed item set 70169/75149
Processed item set 70170/75149
Processed item set 70171/75149
Processed item set 70172/75149
Processed item set 70173/75149
Processed item set 70174/75149
Processed item set 70175/75149
Processed item set 70176/75149
Processed item set 70177/75149
Processed item set 70178/75149
Processed item set 70179/75149
Processed item set 70180/75149
Processed item set 70181/75149
Processed item set 70182/75149
Processed item set 